In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
from huggingface_hub import login
login()

In [3]:
!pip install langchain langchain-community langchain-core
!pip install transformers bitsandbytes torch accelerate
!pip install fastapi uvicorn pyngrok


INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0rc2
    Uninstalling packaging-26.0rc2:
      Successfully uninstalled packaging-26.0rc2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.38.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2

In [4]:
import json
import uuid
import socket
import threading
import time
import random
import re
import asyncio
from typing import List


import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

from pydantic import BaseModel
from fastapi import FastAPI, HTTPException

import uvicorn
from pyngrok import ngrok, conf

from langchain.output_parsers import ResponseSchema, StructuredOutputParser

2026-02-17 16:07:38.297561: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771344458.511756      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771344458.578013      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771344459.107238      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771344459.107275      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771344459.107278      55 computation_placer.cc:177] computation placer alr

# **Model Setup**

In [5]:
MODEL_NAME = "mistralai/Mistral-Nemo-Instruct-2407"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, 
    quantization_config=quant_config,
    device_map="auto"
)

pipe = pipeline(
    "text-generation", 
    model=model, 
    tokenizer=tokenizer, 
    max_new_tokens=1024,
    temperature=0.1,     
    do_sample=True,
    repetition_penalty=1.1
)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Device set to use cuda:0


# **Schemas & Parsers**

In [6]:
single_eval_schema = [
    ResponseSchema(name="score", description="An integer from 0-10."),
    ResponseSchema(name="feedback", description="Constructive critique of the user's answer."),
    ResponseSchema(name="refined_answer", description="The 'Ideal' version of the answer the user should have given."),
    ResponseSchema(name="suggestions", description="Bullet points on technical keywords or concepts to add.")
]
single_parser = StructuredOutputParser.from_response_schemas(single_eval_schema)

summary_schema = [
    ResponseSchema(name="overall_score", description="0-100 total readiness."),
    ResponseSchema(name="strengths", description="List of strings of user strengths."),
    ResponseSchema(name="weaknesses", description="List of strings of user weaknesses."),
    ResponseSchema(name="final_advice", description="A motivating final paragraph.")
]
summary_parser = StructuredOutputParser.from_response_schemas(summary_schema)

# **Routes**

In [7]:
app = FastAPI()
SESSIONS = {}

class StartRequest(BaseModel):
    job_title: str
    num_questions: int

class SubmitRequest(BaseModel):
    session_id: str
    answer: str

class SessionRequest(BaseModel):
    session_id: str


def extract_questions(text: str):
    """Try to extract a JSON list of questions from the LLM output."""
    if not text:
        return None

    text = re.sub(r"```json|```", "", text, flags=re.IGNORECASE)

    list_match = re.search(r"\[[\s\S]*?\]", text)
    if list_match:
        try:
            data = json.loads(list_match.group())
            if isinstance(data, list):
                return data
        except json.JSONDecodeError:
            pass

    obj_match = re.search(r"\{[\s\S]*?\}", text)
    if obj_match:
        try:
            data = json.loads(obj_match.group())
            if isinstance(data, dict) and "questions" in data:
                return data["questions"]
        except json.JSONDecodeError:
            pass

    return None


def is_meaningful(text: str) -> bool:
    clean = text.strip().lower()
    if len(clean) < 5:
        return False
    words = clean.split()
    if len(words) < 2:
        return False
    if len(clean) > 12 and not any(v in clean for v in "aeiou"):
        return False
    return True


async def run_llm_task(prompt: str, max_tokens: int, temp: float = 0.0):
    loop = asyncio.get_event_loop()

    def generate():
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=temp,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        return tokenizer.decode(
            outputs[0][inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        ).strip()

    return await loop.run_in_executor(None, generate)



@app.post("/start")
async def start_interview(req: StartRequest):
    session_id = str(uuid.uuid4())
    num_tech = req.num_questions // 2
    num_soft = req.num_questions - num_tech
    job_subtopics = {
        "Software Engineer": ["system architecture", "scalability", "debugging", "API design"],
        "Frontend Developer": ["UI design", "responsive layout", "frontend performance", "state management"],
        "Backend Developer": ["API design", "database design", "scalability", "security"],
        "Full Stack Developer": ["API design", "UI/UX", "database design", "system architecture"],
        "AI Engineer": ["model deployment", "data pipelines", "scalability", "ML systems"],
        "Data Scientist": ["statistics", "data analysis", "ML models", "feature engineering"],
        "Machine Learning Engineer": ["model optimization", "deployment", "scalability", "data preprocessing"],
        "Data Analyst": ["SQL", "data visualization", "ETL", "business insights"],
        "Data Engineer": ["ETL pipelines", "data warehousing", "scalability", "cloud data systems"],
        "DevOps Engineer": ["CI/CD", "monitoring", "infrastructure as code", "cloud systems"],
        "Cloud Architect": ["cloud design", "scalability", "security", "cost optimization"],
        "Cybersecurity Analyst": ["threat detection", "network security", "incident response"],
        "Mobile App Developer": ["mobile architecture", "performance optimization", "UI/UX"],
        "Game Developer": ["game loops", "performance optimization", "graphics systems"],
        "UI/UX Designer": ["design systems", "user research", "prototyping"],
        "Product Manager": ["requirements analysis", "roadmaps", "stakeholder communication"],
        "QA Automation Engineer": ["test automation", "CI/CD testing", "bug tracking"],
        "Embedded Systems Engineer": ["real-time systems", "hardware interfacing", "low-level programming"],
        "Site Reliability Engineer (SRE)": ["monitoring", "incident response", "scalability"],
        "System Administrator": ["server management", "networking", "security"]
    }

    random_focus = random.choice(job_subtopics.get(req.job_title, ["system architecture", "scalability", "debugging", "API design"]))

    prompt = f"""<s>[INST] <<SYS>> Return ONLY a JSON list of strings. <</SYS>>
Generate {req.num_questions} interview questions for a {req.job_title}.
- {num_tech} Technical ({random_focus})
- {num_soft} Soft skills. [/INST]"""

    gen_text = await run_llm_task(prompt, max_tokens=600)
    final_qs = extract_questions(gen_text)

    if not final_qs:
        final_qs = ["Tell me about your background."]
    final_qs = final_qs[:req.num_questions]

    answers = [""] * len(final_qs)

    SESSIONS[session_id] = {
        "job_title": req.job_title,
        "questions": final_qs,
        "answers": answers,
        "results": []
    }

    return {"session_id": session_id, "questions": final_qs}


@app.post("/submit")
def submit_answer(req: SubmitRequest):
    session = SESSIONS.get(req.session_id)
    if not session:
        raise HTTPException(status_code=404, detail="Session not found.")

    try:
        idx = session["answers"].index("")
        session["answers"][idx] = req.answer
    except ValueError:
        return {"completed": True}

    completed = all(is_meaningful(a) or a == "" for a in session["answers"])
    return {"completed": completed}



@app.post("/evaluate")
async def evaluate(req: SessionRequest):
    session = SESSIONS.get(req.session_id)
    if not session:
        return {"error": "Invalid session"}

    questions = session.get("questions", [])
    answers = session.get("answers", [])
    format_instructions = single_parser.get_format_instructions()

    eval_template = """<s>[INST]
You are a strict Technical Hiring Manager evaluating a candidate for a {role} position.

RULES (MANDATORY):
- If answer is gibberish or meaningless, score MUST be 0–2

SCORING RUBRIC:
- 0–2: Empty, gibberish, irrelevant
- 3–5: Partially correct, lacks depth
- 6–8: Correct with solid understanding
- 9–10: Exceptional; edge cases, trade-offs, best practices

DEFINITIONS:
- feedback: Critical and honest evaluation
- refined_answer: High-quality model answer
- suggestions: Missing keywords or improvements

{format_instructions}

QUESTION:
{q}

CANDIDATE ANSWER:
{a}

IMPORTANT INSTRUCTION:
- If the candidate answer is gibberish, unreadable, or extremely brief, still generate a high-quality model answer in "refined_answer".
- Provide critical and honest "feedback".
- Provide "suggestions" for improvement.
- Score must be 0–2 if the answer is gibberish.
[/INST]"""

    async def evaluate_one(q, a):
        meaningful = is_meaningful(a)

        prompt = eval_template.format(
            role=session.get("job_title", "Unknown Role"),
            q=q,
            a=a if meaningful else "[Candidate answer is gibberish]",
            format_instructions=format_instructions
        )

        raw_gen = await run_llm_task(prompt, max_tokens=450, temp=0.0)
        
        try:
            parsed = single_parser.parse(raw_gen)
        except Exception:
            parsed = {
        "score": 0,
        "feedback": "Failed to parse",
        "refined_answer": "N/A",
        "suggestions": "N/A"
    }

        if not parsed:
            parsed = {"score": 0, "feedback": "Failed to parse", "refined_answer": "N/A", "suggestions": "N/A"}

        if not meaningful:
            parsed["score"] = min(parsed.get("score", 0), 2)
            parsed["feedback"] = "Response detected as gibberish or too brief to evaluate."

        parsed.update({"question": q, "answer": a})
        return parsed

    tasks = [evaluate_one(q, a) for q, a in zip(questions, answers)]
    results = await asyncio.gather(*tasks)
    session["results"] = results
    torch.cuda.empty_cache()
    return {"results": results}



@app.post("/summary")
async def summary(req: SessionRequest):
    session = SESSIONS.get(req.session_id)
    if not session or not session.get("answers"):
        return {"overall_score": 0, "strengths": [], "weaknesses": [], "final_advice": "No answers submitted."}

    format_instructions = summary_parser.get_format_instructions()

    context_list = [f"Q: {q} | A: {a}" for q, a in zip(session["questions"], session["answers"])]
    data_str = "\n".join(context_list)

    summary_template = """<s>[INST] You are a strict senior career coach. 
Evaluate the interview performance for a {job_title} role **strictly based on the candidate's actual answers**. 

{format_instructions}

PERFORMANCE DATA:
{data_str}

Instructions:
1. Assign an "overall_score" 0-100 strictly based on quality, clarity, and technical correctness.
2. List up to 3 **real strengths**. If none, return an empty list [].
3. List up to 3 **real weaknesses**. If none, return an empty list [].
4. Provide "final_advice" in exactly 2 sentences, grounded in the candidate’s actual answers.
5. Do NOT make up skills, examples, or scenarios. If answers are unreadable or irrelevant, indicate that clearly.
[/INST]"""

    prompt = summary_template.format(
        job_title=session['job_title'],
        data_str=data_str,
        format_instructions=format_instructions
    )

    raw_output = await run_llm_task(prompt, max_tokens=300, temp=0.3)
    
    try:
        parsed_sum = summary_parser.parse(raw_output)
    except Exception:
        parsed_sum = {
        "overall_score": 0,
        "strengths": [],
        "weaknesses": ["Error parsing summary"],
        "final_advice": "Summary failed."
    }

    if not parsed_sum:
        return {"overall_score": 0, "strengths": [], "weaknesses": ["Error"], "final_advice": "Summary failed."}

    return parsed_sum

In [38]:
ngrok.kill()

# **Deployment**

In [8]:
if __name__ == "__main__":

    NGROK_TOKEN = "YOUR NGROK TOKEN"
    conf.get_default().auth_token = NGROK_TOKEN

    import socket, threading, time
    def free_port():
        s = socket.socket()
        s.bind(('', 0))
        port = s.getsockname()[1]
        s.close()
        return port

    port = free_port()
    public_url = ngrok.connect(port).public_url
    print(f"🚀 Public API URL: {public_url}")

    def run():
        uvicorn.run(app, host="0.0.0.0", port=port)

    threading.Thread(target=run, daemon=True).start()
    time.sleep(1)

🚀 Public API URL: https://disposed-thomasine-unparching.ngrok-free.dev                             


INFO:     Started server process [55]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:51413 (Press CTRL+C to quit)
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


INFO:     156.203.143.121:0 - "POST /start HTTP/1.1" 200 OK
INFO:     156.203.143.121:0 - "POST /submit HTTP/1.1" 200 OK
INFO:     156.203.143.121:0 - "POST /submit HTTP/1.1" 200 OK
INFO:     156.203.143.121:0 - "POST /submit HTTP/1.1" 200 OK
INFO:     156.203.143.121:0 - "POST /submit HTTP/1.1" 200 OK
INFO:     156.203.143.121:0 - "POST /evaluate HTTP/1.1" 200 OK
INFO:     156.203.143.121:0 - "POST /summary HTTP/1.1" 200 OK
INFO:     156.203.143.121:0 - "POST /start HTTP/1.1" 200 OK
INFO:     156.203.143.121:0 - "POST /submit HTTP/1.1" 200 OK
INFO:     156.203.143.121:0 - "POST /submit HTTP/1.1" 200 OK
INFO:     156.203.143.121:0 - "POST /submit HTTP/1.1" 200 OK
INFO:     156.203.143.121:0 - "POST /submit HTTP/1.1" 200 OK
INFO:     156.203.143.121:0 - "POST /evaluate HTTP/1.1" 200 OK
INFO:     156.203.143.121:0 - "POST /summary HTTP/1.1" 200 OK
